<a href="https://colab.research.google.com/github/shravyataluka-codes/neurons-to-networks/blob/main/classification_ex1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Task 1: Data Ingestion & Cleaning
import pandas as pd

# Load dataset into DataFrame
df = pd.read_csv("/content/WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Check data types
print("Data Types:\n", df.dtypes)

# Identify missing values
print("\nMissing Values:\n", df.isnull().sum())

# Drop redundant constant columns
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])

# Confirm shape after cleaning
print("\nDataset shape after cleaning:", df.shape)


Data Types:
 Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWo

In [3]:
# Task 2: Exploratory Data Analysis & Class Imbalance Check

# Count how many employees left vs stayed
attrition_counts = df["Attrition"].value_counts()

# Calculate percentage distribution
attrition_percent = df["Attrition"].value_counts(normalize=True) * 100

print("Attrition Counts:\n", attrition_counts)
print("\nAttrition Percentage:\n", attrition_percent)


Attrition Counts:
 Attrition
No     1233
Yes     237
Name: count, dtype: int64

Attrition Percentage:
 Attrition
No     83.877551
Yes    16.122449
Name: proportion, dtype: float64


In [4]:
# Task 3: Feature Preprocessing & Stratified Splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

# Separate features and target
X = df.drop(columns=["Attrition"])
y = df["Attrition"].map({"Yes":1, "No":0})

# Identify numerical and categorical features
num_features = X.select_dtypes(include=["int64","float64"]).columns
cat_features = X.select_dtypes(exclude=["int64","float64"]).columns

# Preprocessing: scale numeric, one-hot encode categorical
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

# Stratified train-test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Fit-transform training, transform test
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Apply SMOTE strictly on training set
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


Train shape: (1972, 51) Test shape: (294, 51)


In [5]:
# Task 4: Binary Classification (Predicting Attrition)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Pivot target to Attrition
X = df.drop(columns=["Attrition"])
y = df["Attrition"].map({"Yes":1, "No":0})

# Identify numerical and categorical features
num_features = X.select_dtypes(include=["int64","float64"]).columns
cat_features = X.select_dtypes(exclude=["int64","float64"]).columns

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Transform data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Apply SMOTE on training set
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Train and evaluate models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))
    if name != "Decision Tree":  # ROC-AUC works best with probability outputs
        print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))



Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.91      0.80      0.85       247
           1       0.36      0.57      0.44        47

    accuracy                           0.77       294
   macro avg       0.63      0.69      0.65       294
weighted avg       0.82      0.77      0.79       294

ROC-AUC: 0.7957619088638127

Decision Tree Results:
              precision    recall  f1-score   support

           0       0.87      0.87      0.87       247
           1       0.33      0.34      0.33        47

    accuracy                           0.78       294
   macro avg       0.60      0.60      0.60       294
weighted avg       0.79      0.78      0.78       294


Random Forest Results:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91       247
           1       0.52      0.26      0.34        47

    accuracy                           0.84       294
   macro avg       0.7

In [6]:
# Task 5: Binary Classification (Predicting OverTime)
from sklearn.metrics import confusion_matrix

# Pivot target to OverTime
X = df.drop(columns=["OverTime"])
y = df["OverTime"].map({"Yes":1, "No":0})

# Identify numerical and categorical features
num_features = X.select_dtypes(include=["int64","float64"]).columns
cat_features = X.select_dtypes(exclude=["int64","float64"]).columns

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Transform data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Apply SMOTE on training set
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Train and evaluate models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.82      0.67      0.74       211
           1       0.43      0.63      0.51        83

    accuracy                           0.66       294
   macro avg       0.62      0.65      0.62       294
weighted avg       0.71      0.66      0.67       294

Confusion Matrix:
 [[141  70]
 [ 31  52]]

Decision Tree Results:
              precision    recall  f1-score   support

           0       0.72      0.70      0.71       211
           1       0.29      0.31      0.30        83

    accuracy                           0.59       294
   macro avg       0.51      0.51      0.51       294
weighted avg       0.60      0.59      0.60       294

Confusion Matrix:
 [[148  63]
 [ 57  26]]

Random Forest Results:
              precision    recall  f1-score   support

           0       0.75      0.91      0.82       211
           1       0.49      0.22      0.30        83

    accuracy        

In [7]:
# Task 6: Multi-Class Classification (Predicting JobRole or JobSatisfaction)
from sklearn.metrics import classification_report, confusion_matrix

# Pivot target to JobRole (multi-class)
X = df.drop(columns=["JobRole"])
y = df["JobRole"]   # categorical multi-class target

# Identify numerical and categorical features
num_features = X.select_dtypes(include=["int64","float64"]).columns
cat_features = X.select_dtypes(exclude=["int64","float64"]).columns

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Transform data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Train and evaluate models
models = {
    "Multinomial Logistic Regression": LogisticRegression(max_iter=1000, multi_class="multinomial"),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    # Macro and Micro averaged F1-scores included in classification_report
    print(classification_report(y_test, y_pred, digits=3))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Multinomial Logistic Regression Results:
                           precision    recall  f1-score   support

Healthcare Representative      0.304     0.269     0.286        26
          Human Resources      1.000     1.000     1.000        10
    Laboratory Technician      0.536     0.577     0.556        52
                  Manager      0.667     0.600     0.632        20
   Manufacturing Director      0.381     0.276     0.320        29
        Research Director      0.579     0.688     0.629        16
       Research Scientist      0.609     0.661     0.634        59
          Sales Executive      0.970     1.000     0.985        65
     Sales Representative      1.000     0.941     0.970        17

                 accuracy                          0.673       294
                macro avg      0.672     0.668     0.668       294
             weighted avg      0.665     0.673     0.667       294

Confusion Matrix:
 [[ 7  0  4  1 12  0  2  0  0]
 [ 0 10  0  0  0  0  0  0  0]
 [ 2 

In [8]:
# Task 7: Feature Importance & Business Insights
from sklearn.ensemble import RandomForestClassifier

# Use Attrition as target for turnover analysis
X = df.drop(columns=["Attrition"])
y = df["Attrition"].map({"Yes":1, "No":0})

# Identify numerical and categorical features
num_features = X.select_dtypes(include=["int64","float64"]).columns
cat_features = X.select_dtypes(exclude=["int64","float64"]).columns

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Transform data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Train Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Extract feature importances
importances = rf.feature_importances_
feature_names = preprocessor.get_feature_names_out()
sorted_features = sorted(zip(importances, feature_names), reverse=True)

# Top 3 drivers of turnover
top3 = sorted_features[:3]
print("Top 3 Drivers of Turnover:")
for importance, feature in top3:
    print(f"{feature}: {importance:.4f}")

# HR Recommendation Summary
print("\nHR Recommendation Summary:")
print("1. Focus on the top drivers of attrition identified above.")
print("2. Address issues related to these features (e.g., work-life balance, overtime, compensation).")
print("3. Implement targeted HR policies to improve satisfaction and reduce turnover.")


Top 3 Drivers of Turnover:
num__MonthlyIncome: 0.0765
num__Age: 0.0612
num__TotalWorkingYears: 0.0594

HR Recommendation Summary:
1. Focus on the top drivers of attrition identified above.
2. Address issues related to these features (e.g., work-life balance, overtime, compensation).
3. Implement targeted HR policies to improve satisfaction and reduce turnover.


Mounted at /content/drive
